# Review 2: full clean static Transformer training

This notebook trains **from scratch on all 6,417,308 naturally distributed training rows** of the previously prepared combined flow collection. It uses the frozen **static** preprocessing state, full clean validation for checkpoint selection, and opens the complete test partition only after selection. This is the first full-data clean baseline—not a continual, replay, novelty, or poisoning run. Per-row source provenance is unverified; row-level splits cannot rule out duplicate-flow or capture-session dependence.

Enable Kaggle **Internet** and **T4 GPU**. The prepared folder may remain in the current `/kaggle/working` session or be attached as a private Kaggle input. Set `PREPARED_DIR` explicitly below if discovery finds zero or multiple candidates. Prepared arrays are opened read-only and never copied or re-prepared. This notebook may take multiple Kaggle sessions: after each completed epoch, save the `/kaggle/working` run folder as notebook output. To resume, attach that output and set `RESUME_FROM` below. An interrupted partial epoch repeats from the latest completed epoch; optimizer and RNG never silently restart.


## 1. Clone the exact published API and run focused synthetic checks

The pinned branch commit contains durable epoch-boundary resume and bounded confusion metrics. Keep Kaggle's CUDA-enabled PyTorch. These tests are functional checks, not NIDS results; zero discovered tests or any skip/failure stops the notebook.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, platform, re, random
from dataclasses import asdict
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '91a26a1c13a563a7628e217f357dc27f73874784'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-full-clean')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.data import BROAD_CLASS_ORDER, FittedPreprocessor
from src.data.large_data import large_data_paths, verify_full_partitions
from src.training.disk_backed import DiskBackedFlowDataset
from src.training.full_run import train_full_disk_backed
from src.evaluation.streaming import evaluate_disk_backed
from src.models import ModelConfig, TabularTransformerClassifier
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable Kaggle T4 GPU and restart')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
environment = {'python': platform.python_version(), 'numpy': version('numpy'), 'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Verified source:', checked_out_sha, 'environment:', environment)
for pattern in ('test_full_disk_training.py', 'test_streaming_evaluation.py', 'test_large_data.py'):
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern, '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
    print(result.stdout, end=''); print(result.stderr, end='')
    summary = result.stdout + result.stderr
    count = re.search(r'Ran (\d+) tests? in', summary)
    if count is None or int(count.group(1)) == 0 or re.search(r'OK \(skipped=\d+\)', summary):
        raise RuntimeError(f'Focused tests absent or skipped: {pattern}')


## 2. Select the already prepared folder and verify its identity

`PREPARED_DIR` is the explicit selector. Use the printed `/kaggle/working/review2_large_prepare_...` folder in the current session, or attach the whole saved folder as a private Kaggle dataset and set its `/kaggle/input/...` path. Discovery is only a convenience; multiple matches stop for an explicit choice. The preparation source/input SHA, completed marker, split seed, class mapping, feature order, static fit scope, shapes/dtypes, file sizes, and complete split coverage must all agree. No Parquet rescan or substitution with the 2,000-per-class sample is allowed.


In [ ]:
PREPARED_DIR = None  # set explicitly, e.g. '/kaggle/working/review2_large_prepare_...' or '/kaggle/input/.../review2_large_prepare_...'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
EXPECTED_PREP_SOURCE_SHA = '5c0d519596bc6c4b99948126443635c2f7eb15f1'
EXPECTED_ROWS = 9_167_581
EXPECTED_SPLIT_ROWS = {'train': 6_417_308, 'validation': 1_375_139, 'test': 1_375_134}
if PREPARED_DIR is None:
    candidates = [p.parent for base in (Path('/kaggle/working'), Path('/kaggle/input')) for p in base.rglob('manifest.json') if 'review2_large_prepare_' in str(p.parent)]
    print('Prepared candidates:', [str(p) for p in candidates])
    if len(candidates) != 1:
        raise RuntimeError(f'Found {len(candidates)} prepared folders; set PREPARED_DIR explicitly')
    prepared_dir = candidates[0]
else:
    prepared_dir = Path(PREPARED_DIR)
if not prepared_dir.is_dir() or not (prepared_dir.is_relative_to(Path('/kaggle/working')) or prepared_dir.is_relative_to(Path('/kaggle/input'))):
    raise ValueError('PREPARED_DIR must be a prepared Kaggle working/input folder')
prep_manifest = json.loads((prepared_dir / 'manifest.json').read_text())
split_manifest = json.loads((prepared_dir / 'split_manifest.json').read_text())
features_manifest = json.loads((prepared_dir / 'features_manifest.json').read_text())
static_record = json.loads((prepared_dir / 'preprocessing_static.json').read_text())
continual_record = json.loads((prepared_dir / 'preprocessing_continual.json').read_text())
assert prep_manifest['completed_preparation_and_benchmark_only'] is True
assert prep_manifest['source_sha'] == EXPECTED_PREP_SOURCE_SHA and prep_manifest['input_sha256'] == EXPECTED_INPUT_SHA256
assert prep_manifest['input_rows'] == split_manifest['row_count'] == features_manifest['row_count'] == EXPECTED_ROWS
assert split_manifest['seed'] == 42 and split_manifest['fractions'] == [0.70, 0.15, 0.15] and split_manifest['class_order'] == list(BROAD_CLASS_ORDER)
assert split_manifest['split_codes'] == {'train': 0, 'validation': 1, 'test': 2}
assert features_manifest['dtype'] == 'float32' and features_manifest['targets_excluded'] == ['Label', 'ClassLabel']
assert len(features_manifest['feature_names']) == 54 and features_manifest['feature_names'] == prep_manifest['stages']['feature_store']['feature_names']
assert static_record['scope'] == 'static' and continual_record['scope'] == 'continual'
assert static_record['feature_names'] == continual_record['feature_names'] == features_manifest['feature_names']
assert static_record['fit_rows'] == EXPECTED_SPLIT_ROWS['train'] and static_record['reservoir_seed'] == 42 and static_record['reservoir_size_used'] == 200_000
assert continual_record['fit_rows'] == sum(split_manifest['counts'][name]['train'] for name in BROAD_CLASS_ORDER[:5])
paths = large_data_paths(prepared_dir)
actual_split = verify_full_partitions(paths, EXPECTED_ROWS)
assert actual_split['counts'] == split_manifest['counts']
assert {part: sum(row[part] for row in actual_split['counts'].values()) for part in EXPECTED_SPLIT_ROWS} == EXPECTED_SPLIT_ROWS
expected_arrays = {'raw_features.npy': ('float32', (EXPECTED_ROWS, 54)), 'labels.npy': ('uint8', (EXPECTED_ROWS,)), 'splits.npy': ('uint8', (EXPECTED_ROWS,)), 'row_ids.npy': ('int64', (EXPECTED_ROWS,))}
for filename, (dtype, shape) in expected_arrays.items():
    array = np.load(prepared_dir / filename, mmap_mode='r', allow_pickle=False)
    assert array.dtype == np.dtype(dtype) and array.shape == shape
    assert (prepared_dir / filename).stat().st_size == prep_manifest['artifact_bytes'][filename]
for part, count in EXPECTED_SPLIT_ROWS.items():
    array = np.load(prepared_dir / f'{part}_indices.npy', mmap_mode='r', allow_pickle=False)
    assert array.dtype == np.int64 and array.shape == (count,)
static_state = FittedPreprocessor(tuple(static_record['feature_names']), np.asarray(static_record['imputation_values'], dtype=np.float64), np.asarray(static_record['means'], dtype=np.float64), np.asarray(static_record['scales'], dtype=np.float64))
assert all(np.isfinite(values).all() for values in (static_state.imputation_values, static_state.means, static_state.scales)) and np.all(static_state.scales > 0)
print('Validated read-only prepared folder:', prepared_dir, 'split sizes:', EXPECTED_SPLIT_ROWS)


## 3. Bind exact prepared-data identity and create a separate run folder

Hash the persisted raw feature store, labels, split codes, static state, and split manifest to bind later resumes to the exact prepared bytes; this reads files but does not alter them. The run folder belongs under `/kaggle/working`, separate from `PREPARED_DIR`. On a new Kaggle session, attach a previously saved run folder and set `RESUME_FROM`; its small checkpoint/history files are copied into a new writable run folder. Never copy or regenerate the 2.1-GB prepared arrays. If continuing in the same working session, set `RESUME_FROM` to that existing run folder. Save a Kaggle version at least after each completed epoch.


In [ ]:
RESUME_FROM = None  # optional existing '/kaggle/working/full_clean_...' or attached '/kaggle/input/.../full_clean_...' run folder
MAX_EPOCHS_THIS_CALL = None  # optional positive integer to pause intentionally at an epoch boundary
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
hash_start = perf_counter()
data_identity = {'prep_source_sha': prep_manifest['source_sha'], 'input_sha256': prep_manifest['input_sha256'], 'row_count': EXPECTED_ROWS, 'split_seed': 42, 'raw_features_sha256': file_sha256(paths.raw_features), 'labels_sha256': file_sha256(paths.labels), 'splits_sha256': file_sha256(paths.splits), 'static_preprocessing_sha256': file_sha256(prepared_dir / 'preprocessing_static.json'), 'split_manifest_sha256': file_sha256(prepared_dir / 'split_manifest.json'), 'train_rows': EXPECTED_SPLIT_ROWS['train'], 'validation_rows': EXPECTED_SPLIT_ROWS['validation'], 'test_rows': EXPECTED_SPLIT_ROWS['test']}
print('Prepared-byte identity hashed in seconds:', perf_counter() - hash_start)
if RESUME_FROM is None:
    run_dir = Path('/kaggle/working') / ('full_clean_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    run_dir.mkdir(exist_ok=False)
    resume = False
else:
    previous_dir = Path(RESUME_FROM)
    if not previous_dir.is_dir() or not (previous_dir.is_relative_to(Path('/kaggle/working')) or previous_dir.is_relative_to(Path('/kaggle/input'))):
        raise ValueError('RESUME_FROM must be an existing Kaggle working/input run folder')
    if previous_dir.is_relative_to(Path('/kaggle/working')):
        run_dir = previous_dir
    else:
        run_dir = Path('/kaggle/working') / ('full_clean_resume_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        run_dir.mkdir(exist_ok=False)
        for filename in ('latest.pt', 'best.pt', 'history.json', 'training_manifest.json', 'run_config.json', 'run_environment.json', 'reload_verification.json', 'test_metrics.json', 'training_curves.png', 'confusion_matrix.png', 'final_manifest.json'):
            source = previous_dir / filename
            if source.exists():
                shutil.copy2(source, run_dir / filename)
    resume = True
assert run_dir != prepared_dir and not run_dir.is_relative_to(prepared_dir)
if shutil.disk_usage('/kaggle/working').free < 1_000_000_000:
    raise RuntimeError('Less than 1 GB free for durable checkpoints and outputs')
print('Run folder:', run_dir, 'resume:', resume)


## 4. Freeze the clean static configuration and full partition views

Use the full natural-distribution train and validation partitions with the frozen **static** state. Global class IDs remain 0–7. The eight-output Transformer is initialized afresh (unless restoring `latest.pt`); no balanced-sample or continual checkpoint is loaded. Config: hidden64, 4 heads, 2 layers, MLP128, batch256, learning rate 0.001, seed42, mixed precision off, at most 20 epochs, validation-loss patience5, ordinary shuffled order. Bounded views hold row IDs and read-only memory maps, not full transformed matrices.


In [ ]:
CLASS_IDS = tuple(range(8))
config = ModelConfig(num_features=54, num_classes=8, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=256, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42, training_sampler='shuffled')
assert config.training_sampler == 'shuffled' and config.mixed_precision is False
train_view = DiskBackedFlowDataset(prepared_dir, static_state, partition='train', class_ids=CLASS_IDS)
validation_view = DiskBackedFlowDataset(prepared_dir, static_state, partition='validation', class_ids=CLASS_IDS)
assert len(train_view) == EXPECTED_SPLIT_ROWS['train'] and len(validation_view) == EXPECTED_SPLIT_ROWS['validation']
assert np.intersect1d(train_view.indices, validation_view.indices).size == 0
config_record = {'source_sha': checked_out_sha, 'model': asdict(config), 'class_ids': list(CLASS_IDS), 'preprocessing_scope': 'static full training only', 'data_identity': data_identity, 'checkpoint_selection': 'minimum full validation loss only'}
config_path = run_dir / 'run_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == config_record
else:
    config_path.write_text(json.dumps(config_record, indent=2))
env_path = run_dir / 'run_environment.json'
if env_path.exists():
    assert json.loads(env_path.read_text()) == environment
else:
    env_path.write_text(json.dumps(environment, indent=2))
random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
model = TabularTransformerClassifier(config, class_ids=CLASS_IDS)
print('Natural train/validation rows:', len(train_view), len(validation_view), 'class IDs:', model.class_ids)


## 5. Train all training rows with full validation and durable epoch checkpoints

The trainer prints measured batch throughput and training time remaining, then full-epoch train/validation losses, timings, rows and optimizer steps, plus an observed remaining-epoch estimate. `latest.pt` and `best.pt` are saved atomically after **every completed epoch**, along with history and a manifest. Latest is authoritative: it stores optimizer/scaler, model, Python/NumPy/Torch CPU/CUDA RNG, best state, completed epoch and patience. Resume verifies config, class IDs, prepared-byte identity and runtime. A crash within an epoch repeats that epoch from its prior boundary; it never silently resets the optimizer. Save the run folder between Kaggle sessions. Do not open test rows yet.


In [ ]:
training_result = train_full_disk_backed(model, train_view, validation_view, output_dir=run_dir, data_identity=data_identity, resume=resume, max_epochs_this_call=MAX_EPOCHS_THIS_CALL, progress_every_batches=1000, report=lambda message: print(message, flush=True))
print('Completed epoch:', training_result['completed_epoch'], 'best validation epoch:', training_result['best_epoch'], 'early stopped:', training_result['early_stopped'], 'paused:', training_result['paused'])
print('Cumulative training rows:', training_result['history']['training_rows_processed'], 'validation rows:', training_result['history']['validation_rows_processed'], 'optimizer steps:', training_result['history']['optimizer_steps'])
if training_result['paused']:
    raise RuntimeError('Training paused at a completed epoch. Save the run folder, set RESUME_FROM, and rerun before test evaluation.')


## 6. Verify the best checkpoint before opening test rows

Selection used **full clean validation loss only**. Confirm that the loadable best checkpoint exactly matches the best-state snapshot in authoritative latest and that reloaded models give identical bounded validation predictions. Plot training and full-validation curves. No test results influence this choice.


In [ ]:
best_path = run_dir / 'best.pt'; latest_path = run_dir / 'latest.pt'
latest = torch.load(latest_path, map_location='cpu', weights_only=False)
assert latest['best_epoch'] == training_result['best_epoch'] and latest['data_identity'] == data_identity
best_model = TabularTransformerClassifier.load(best_path, device='cuda')
second_load = TabularTransformerClassifier.load(best_path, device='cuda')
assert best_model.class_ids == CLASS_IDS
for name, tensor in best_model.network.state_dict().items():
    assert torch.equal(tensor.detach().cpu(), latest['best_state_dict'][name])
sample_x, _, _ = validation_view.batch(validation_view.indices[:256])
assert np.array_equal(best_model.predict(sample_x), second_load.predict(sample_x))
reload_record = {'best_epoch': training_result['best_epoch'], 'latest_best_state_equal': True, 'bounded_validation_predictions_identical': True, 'best_checkpoint_sha256': file_sha256(best_path)}
(run_dir / 'reload_verification.json').write_text(json.dumps(reload_record, indent=2))
rows = training_result['history']['epochs']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([row['epoch'] for row in rows], [row['train_loss'] for row in rows], label='Training')
axes[0].plot([row['epoch'] for row in rows], [row['validation_loss'] for row in rows], label='Full validation')
axes[0].axvline(training_result['best_epoch'], color='gray', linestyle='--', label='Best')
axes[0].set(xlabel='Epoch', ylabel='Cross-entropy', title='Full clean training/validation loss'); axes[0].legend()
axes[1].plot([row['epoch'] for row in rows], [row['validation_accuracy'] for row in rows], marker='o')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Full validation accuracy')
fig.tight_layout(); fig.savefig(run_dir / 'training_curves.png', dpi=160); plt.show()
print('Best checkpoint/reload verified:', reload_record)


## 7. Evaluate the selected model exactly once on all test rows

Only now open the complete untouched test partition. Inference is bounded by batch256 and retains an 8×8 confusion matrix, not all predictions. Report accuracy, macro-F1, balanced accuracy, every class's precision/recall/F1/support, and Benign FPR. If a verified test record already exists on rerun, reuse it rather than evaluating or tuning again. Row-level splitting and combined-source provenance remain limitations.


In [ ]:
metrics_path = run_dir / 'test_metrics.json'
if metrics_path.exists():
    test_record = json.loads(metrics_path.read_text())
    assert test_record['data_identity'] == data_identity and test_record['best_checkpoint_sha256'] == reload_record['best_checkpoint_sha256']
    print('Reusing existing selected-checkpoint test result; no new test inference')
else:
    test_view = DiskBackedFlowDataset(prepared_dir, static_state, partition='test', class_ids=CLASS_IDS)
    assert len(test_view) == EXPECTED_SPLIT_ROWS['test']
    torch.cuda.synchronize(); start = perf_counter()
    metrics = evaluate_disk_backed(best_model, test_view, batch_size=256, class_names=dict(enumerate(BROAD_CLASS_ORDER)), benign_class_id=0)
    torch.cuda.synchronize(); test_seconds = perf_counter() - start
    assert metrics['total_rows'] == EXPECTED_SPLIT_ROWS['test'] and sum(item['support'] for item in metrics['per_class'].values()) == EXPECTED_SPLIT_ROWS['test']
    test_record = {'data_identity': data_identity, 'best_checkpoint_sha256': reload_record['best_checkpoint_sha256'], 'best_validation_epoch': training_result['best_epoch'], 'test_seconds': test_seconds, 'metrics': metrics}
    metrics_path.write_text(json.dumps(test_record, indent=2))
metrics = test_record['metrics']
print('Full test accuracy:', metrics['accuracy'], 'macro-F1:', metrics['macro_f1'], 'balanced accuracy:', metrics['balanced_accuracy'], 'Benign FPR:', metrics['benign_false_positive']['rate'])
for class_id in CLASS_IDS:
    print(class_id, metrics['per_class'][str(class_id)])
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(np.asarray(metrics['confusion_matrix']), cmap='Blues')
ax.set_xticks(range(8), BROAD_CLASS_ORDER, rotation=45, ha='right'); ax.set_yticks(range(8), BROAD_CLASS_ORDER)
ax.set(xlabel='Predicted broad class', ylabel='True broad class', title='Full clean test confusion counts')
fig.colorbar(image, ax=ax); fig.tight_layout(); fig.savefig(run_dir / 'confusion_matrix.png', dpi=160); plt.show()


## 8. Save the final audit record and preserve outputs

The training folder contains per-epoch `latest.pt`/`best.pt`, `history.json`, `training_manifest.json`, source/config/environment/data identity, reload verification, test metrics, and plots. Save this `/kaggle/working/full_clean_...` folder as a Kaggle version output or private dataset before the session expires. Prepared arrays remain separate and read-only. Send the files listed below back for review; do not interpret this notebook as continual or poisoning evidence. Subsequent full-data continual/replay and poisoning comparisons require separate runs.


In [ ]:
final_manifest = {'status': 'completed full clean static baseline', 'source_sha': checked_out_sha, 'prepared_dir': str(prepared_dir), 'data_identity': data_identity, 'config_file': 'run_config.json', 'environment_file': 'run_environment.json', 'completed_epoch': training_result['completed_epoch'], 'best_epoch': training_result['best_epoch'], 'best_validation_loss': training_result['best_validation_loss'], 'early_stopped': training_result['early_stopped'], 'test_rows_evaluated_once': metrics['total_rows'], 'reload_verified': True, 'artifact_names': sorted(path.name for path in run_dir.iterdir() if path.is_file())}
(run_dir / 'final_manifest.json').write_text(json.dumps(final_manifest, indent=2))
print('Completed run folder:', run_dir)
print('Send executed notebook and final_manifest.json, training_manifest.json, history.json, run_config.json, run_environment.json, reload_verification.json, test_metrics.json, training_curves.png, confusion_matrix.png. Preserve best.pt/latest.pt privately for audit and resume; do not commit them.')
